In [ ]:
import json
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions
import nltk
from groq import Groq
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key=""
    )


# -------------------------------------------------
# PDF Processing & Chunking
# -------------------------------------------------
reader = PdfReader("Practice.pdf")

full_text = ""
for page in reader.pages:
    full_text += page.extract_text()

sentences = nltk.sent_tokenize(full_text)


def group_chunks(sentences, group_size=4):
    chunked_sentences = []
    for i in range (0, len(sentences), group_size):
        batch = sentences[i:i+group_size]
        chunked_sentences.append(" ".join(batch))
    return chunked_sentences


chunks = group_chunks(sentences)

# 1. Create a client (in-memory or persistent on disk)
client = chromadb.PersistentClient(path="./chroma_db")

# added step - using custom embedding model - all-MiniLM-L6-v2
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-v2"
)

# 2. Create a "collection" — think of it like a table, but for vectors
collection = client.get_or_create_collection(
    name="Practice_Collection",
    embedding_function = sentence_transformer_ef
    
    )


if collection.count() == 0:
    collection.add(
    documents=chunks,
        ids=[str(i) for i in range(len(chunks))]
    )


# -------------------------------------------------
#     LOAD AND LOOP QUESTIONS
# -------------------------------------------------
with open ('eval_set.json', 'r') as file:
    eval_set = json.load(file)

eval_results = []
for entry in eval_set:
    question = entry["question"]
    
    results = collection.query(
        query_texts=[question],
        n_results=5
    )

    retrieved_chunks = results["documents"][0]

    # -----------------------------------------------
    #       LOAD GROQ API
    # -----------------------------------------------

    context = "\n\n".join(retrieved_chunks)

    prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the context below.\n\nContext:\n{context}"),
    ("human", "{question}")
    ])


    chain = prompt | model | StrOutputParser()

    response = chain.invoke({"context": context, "question": question})


    print(response)


    eval_results.append({
        "id": entry["id"],
        "question": question,
        "ground_truth_answer": entry["ground_truth_answer"],
        "source_section": entry["source_section"],
        "retrieved_chunks": [
            {"chunk_id": cid, "text": txt}
            for cid, txt in zip(results["ids"][0], results["documents"][0])
        ],
        "generated_answer": response
    })

with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

/home/dishant/anaconda3/envs/ai-engineering/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3585.02it/s]


The survey actually only mentions one high-level template for applying deep learning in scientific settings, which is:

(1) Prediction Problems: using deep learning to tackle important prediction problems, such as mapping inputs to predicted outputs. For example, using a biopsy image as input to predict whether the imaged tissue shows signs of cancer.

The survey does not explicitly mention two other templates. It does discuss various deep learning concepts, tasks, and techniques, but it only provides one example of a high-level template for applying deep learning in scientific settings.
According to the context, what distinguishes instance segmentation from semantic segmentation is that instance segmentation categorizes each pixel separately for each instance of an object, and also provides instance-specific bounding boxes, whereas semantic segmentation only categorizes pixels according to their high-level group (e.g. "pedestrian", "vehicle", "road") without differentiating between in

In [ ]:
import json
import os
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=""
)

with open('eval_results.json', 'r') as file:
    eval_results = json.load(file)


def precision_at_k(retrieved_chunks, source_section):
    keyword = source_section.split()[-1].lower()
    hits = sum(1 for chunk in retrieved_chunks if keyword in chunk["text"].lower())
    return hits / len(retrieved_chunks)


def llm_judge_faithfulness(context, answer, model):
    prompt = f"""Score faithfulness 0.0 to 1.0.
Faithfulness = answer only uses info from context.

Context: {context}
Answer: {answer}

Respond ONLY with valid JSON: {{"faithfulness": <score>}}"""
    response = model.invoke(prompt)
    scores = json.loads(response.content)
    return scores["faithfulness"]


def llm_judge_relevance(question, answer, model):
    prompt = f"""Score answer relevance 0.0 to 1.0.
Relevance = answer directly addresses the question.

Question: {question}
Answer: {answer}

Respond ONLY with valid JSON: {{"answer_relevance": <score>}}"""
    response = model.invoke(prompt)
    scores = json.loads(response.content)
    return scores["answer_relevance"]


results = []
for entry in eval_results:
    context = "\n\n".join([c["text"] for c in entry["retrieved_chunks"]])

    p_at_k = precision_at_k(entry["retrieved_chunks"], entry["source_section"])
    faith = llm_judge_faithfulness(context, entry["generated_answer"], model)
    relevance = llm_judge_relevance(entry["question"], entry["generated_answer"], model)

    results.append({
        "id": entry["id"],
        "question": entry["question"],
        "precision_at_k": p_at_k,
        "faithfulness": faith,
        "answer_relevance": relevance
    })

    print(f"{entry['id']} | precision@k: {p_at_k:.2f} | faithfulness: {faith:.2f} | relevance: {relevance:.2f}")

with open("eval_scores.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nScores written to eval_scores.json")

eval_001 | precision@k: 0.20 | faithfulness: 0.00 | relevance: 0.00
eval_002 | precision@k: 1.00 | faithfulness: 1.00 | relevance: 1.00
eval_003 | precision@k: 0.80 | faithfulness: 0.80 | relevance: 0.80
eval_004 | precision@k: 0.40 | faithfulness: 1.00 | relevance: 1.00
eval_005 | precision@k: 1.00 | faithfulness: 1.00 | relevance: 1.00
eval_006 | precision@k: 0.80 | faithfulness: 1.00 | relevance: 1.00
eval_007 | precision@k: 0.80 | faithfulness: 0.80 | relevance: 1.00
eval_008 | precision@k: 0.80 | faithfulness: 1.00 | relevance: 1.00
eval_009 | precision@k: 0.20 | faithfulness: 1.00 | relevance: 0.80
eval_010 | precision@k: 0.20 | faithfulness: 1.00 | relevance: 0.20

Scores written to eval_scores.json


In [7]:
with open ('eval_scores.json', 'r') as file:
    eval_scores = json.load(file)

eval_scores

[{'id': 'eval_001',
  'question': 'What are the three high-level templates the survey describes for applying deep learning in scientific settings?',
  'precision_at_k': 0.2,
  'faithfulness': 0.0,
  'answer_relevance': 0.0},
 {'id': 'eval_002',
  'question': 'What distinguishes instance segmentation from semantic segmentation according to the survey?',
  'precision_at_k': 1.0,
  'faithfulness': 1.0,
  'answer_relevance': 1.0},
 {'id': 'eval_003',
  'question': 'Why does the survey recommend graph neural networks for chemistry tasks?',
  'precision_at_k': 0.8,
  'faithfulness': 0.8,
  'answer_relevance': 0.8},
 {'id': 'eval_004',
  'question': 'What problem does the Transformer architecture solve that RNNs struggled with?',
  'precision_at_k': 0.4,
  'faithfulness': 1.0,
  'answer_relevance': 1.0},
 {'id': 'eval_005',
  'question': 'What is the two-step process of transfer learning as described in the survey?',
  'precision_at_k': 1.0,
  'faithfulness': 1.0,
  'answer_relevance': 1.0},


In [8]:
import json
import os
from datetime import datetime

BASELINE_FILE = "baseline_scores.json"
HISTORY_FILE = "score_history.json"
DRIFT_THRESHOLD = 0.2


def compute_means(scores):
    n = len(scores)
    return {
        "precision_at_k": sum(s["precision_at_k"] for s in scores) / n,
        "faithfulness": sum(s["faithfulness"] for s in scores) / n,
        "answer_relevance": sum(s["answer_relevance"] for s in scores) / n
    }


# Load current scores
with open("eval_scores.json", "r") as f:
    current_scores = json.load(f)

current_means = compute_means(current_scores)
print(f"Current run means: {current_means}")

# If no baseline exists, save current as baseline and exit
if not os.path.exists(BASELINE_FILE):
    with open(BASELINE_FILE, "w") as f:
        json.dump(current_means, f, indent=2)
    print("No baseline found. Current scores saved as baseline.")
else:
    # Load baseline and compare
    with open(BASELINE_FILE, "r") as f:
        baseline = json.load(f)

    print(f"Baseline means:     {baseline}")

    drift_detected = False
    for metric in ["precision_at_k", "faithfulness", "answer_relevance"]:
        drop = baseline[metric] - current_means[metric]
        if drop > DRIFT_THRESHOLD:
            print(f"DRIFT WARNING: {metric} dropped by {drop:.2f} (baseline: {baseline[metric]:.2f} → current: {current_means[metric]:.2f})")
            drift_detected = True

    if not drift_detected:
        print("No drift detected. Scores are within threshold.")

# Append current run to history
run_entry = {
    "timestamp": datetime.utcnow().isoformat(),
    "means": current_means
}

if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
else:
    history = []

history.append(run_entry)

with open(HISTORY_FILE, "w") as f:
    json.dump(history, f, indent=2)

print(f"Run logged to {HISTORY_FILE}.")

Current run means: {'precision_at_k': 0.62, 'faithfulness': 0.86, 'answer_relevance': 0.78}
No baseline found. Current scores saved as baseline.
Run logged to score_history.json.
